### Imports

In [ ]:
experiment_name = 'Saccharose hydrolysis'

In [ ]:
import sys
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import pulp
import pickle
import time

In [ ]:
import json

with open('../config.json', 'r') as f:
    config = json.load(f)

MAGNETSTEIN_PATH = config["magnetstein_path"]
MIXTURE_PATH_INT = config["mixture_paths_int"][experiment_name]
#(it"s ok that we are using preprocessed_mixture for PMG 287 
#cause the only preprocessing here is zeros from edges i.e. taking region: (2.952301, 10.387961))
MIXTURE_SEPARATOR = config["mixture_separators"][experiment_name]
PYTHON_INTEGRALS_PATH = config["python_integrals_paths"][experiment_name]
SUBSTANCES_NAMES = config["substances_names"][experiment_name]
INTEGRATION_INTERVALS = config["integration_intervals"][experiment_name]
RESULT_PATH = config["results_paths"][experiment_name]

In [ ]:
import sys
sys.path.insert(0, MAGNETSTEIN_PATH)
from masserstein import NMRSpectrum, estimate_proportions

### Data

#### Mixture in time

In [ ]:
mixture_time_data = pd.read_csv(MIXTURE_PATH_INT, sep = MIXTURE_SEPARATOR)
if experiment_name == 'Saccharose hydrolysis':
    ppm = mixture_time_data.iloc[:,0:1]
    ints = mixture_time_data.iloc[:,1:-1]
    mixture_time_data = pd.concat((ppm, ints), axis=1)
elif experiment_name == 'PMG 284 monitoring':
    ppm = mixture_time_data.iloc[:,:-2].iloc[:,0:1]
    ints = mixture_time_data.iloc[:,:-2].iloc[:,1:]
    mixture_time_data = pd.concat((ppm, ints), axis=1)
elif experiment_name == 'PMG 287 monitoring':
    ppm = mixture_time_data.iloc[:,:-1].iloc[:,0:1]
    ints = mixture_time_data.iloc[:,:-1].iloc[:,1:]
    mixture_time_data = pd.concat((ppm, ints), axis=1)

In [ ]:
names = ['ppm'] + ['t' + str(nb) for nb in range(1, mixture_time_data.shape[1])]

In [ ]:
mixture_time_data.columns = names

In [ ]:
def load_spectrum(mixture_time_data, moment_of_time):
    ppm = mixture_time_data['ppm']
    intensity = mixture_time_data['t'+str(moment_of_time)]
    sp = NMRSpectrum(confs = list(zip(ppm, intensity)))
    sp.trim_negative_intensities()
    sp.normalize()
    return sp

In [ ]:
#load_spectrum(mixture_time_data, 999).plot(profile=True)

Note that no baseline correction is needed for saccharose hydrolysis anymore!

### Integrals changing in time

In [ ]:
data_cut_to_intervals = []
for interval in INTEGRATION_INTERVALS:
    data_in_interval = mixture_time_data[
                                        mixture_time_data['ppm'].apply(lambda x:
                                                                               x>interval[0] and x<interval[1])
                                        ]
    data_cut_to_intervals.append(data_in_interval)
    
if experiment_name == 'PMG 284 monitoring':
    data_cut_to_intervals[2] = data_cut_to_intervals[2].fillna(0.)

integrals_changing_in_time = []
    
for timepoint in ['t' + str(nb) for nb in range(1, mixture_time_data.shape[1])]:
    
    integrals_fixed_time = []
    
    for data_in_interval in data_cut_to_intervals:
        
        x_fixed_interval = data_in_interval['ppm']
        y_fixed_interval = data_in_interval[timepoint]
        
        new_int = np.trapz(y = y_fixed_interval, x = x_fixed_interval)
        integrals_fixed_time.append(new_int)
        
    integral_entire_spectrum = np.trapz(y = mixture_time_data[timepoint], x = mixture_time_data['ppm'])
    integrals_fixed_time.append(integral_entire_spectrum)
    
    
    integrals_changing_in_time.append(integrals_fixed_time)

In [ ]:
python_integrals = np.array(integrals_changing_in_time)

In [ ]:
if experiment_name == 'PMG 287 monitoring':
    hexene_integral = python_integrals[:,:5].sum(axis=1).reshape(-1,1)
    triethylsilane_integral = python_integrals[:,5:8].sum(axis=1).reshape(-1,1)
    product_integral = python_integrals[:,8:12].sum(axis=1).reshape(-1,1)
    whole_integral = python_integrals[:,12:13].sum(axis=1).reshape(-1,1)
    python_integrals = np.concatenate([hexene_integral, triethylsilane_integral, product_integral, whole_integral],
                                     axis=1)

### Figures

In [ ]:
for i in range(python_integrals.shape[1]-1):
    plt.plot(python_integrals[:,i] / python_integrals[:, :-1].sum(1), 'p')

### Saving results

In [ ]:
colnames = [name + ': ' + str(interval) for name, interval in zip(SUBSTANCES_NAMES,
                                                                INTEGRATION_INTERVALS)
           ]

In [ ]:
python_integrals_df = pd.DataFrame(python_integrals, columns=colnames + ['whole_spectrum'])

In [ ]:
python_integrals_df.to_csv(PYTHON_INTEGRALS_PATH +
                           'python_integral_' + 
                           '_'.join(experiment_name.split()) + 
                           '.csv',
                          index=False)